In [1]:
import os
import sys
sys.path.append(os.path.abspath('..'))

from dotenv import load_dotenv
from pathlib import Path
from utils.utils import sliding_windows
import joblib
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import zscore

load_dotenv('../.env')

True

In [2]:
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

BASE_PATH = os.getenv("BASE_PATH")
PICKLE_PATH = BASE_PATH + os.getenv("PICKLE_PATH")

In [3]:
from scipy.signal import butter, filtfilt

WINDOWS_SETUP=[
    [1, 0.5],
    [1, 1],
    [1.5, 0.5],
    [1.5, 1],
    [1.5, 1.5],
    [2, 0.5],
    [2, 1],
    [2, 1.5],
    [2, 2]
]

eo_data = joblib.load(PICKLE_PATH + 'eo_crop.pkl')
ec_data = joblib.load(PICKLE_PATH + 'ec_crop.pkl')


def split_raw_by_time(raw_list, train_ratio=0.8, val_ratio=0.1):
    train_raw, val_raw, test_raw = [], [], []

    for raw in raw_list:
        sfreq = raw.info['sfreq']
        n_times = raw.n_times
        train_end = int(n_times * train_ratio)
        val_end = int(n_times * (train_ratio + val_ratio))

        train_raw.append(raw.copy().crop(tmin=0, tmax=(train_end - 1) / sfreq))
        val_raw.append(raw.copy().crop(tmin=train_end / sfreq, tmax=(val_end - 1) / sfreq))
        test_raw.append(raw.copy().crop(tmin=val_end / sfreq, tmax=(n_times - 1) / sfreq))

    return train_raw, val_raw, test_raw

# Butterworth Bandpass Filter (Order 5, 4-40 Hz)
def butter_bandpass_filter(data, lowcut=4, highcut=40, fs=160, order=5):
    """
    Apply Butterworth bandpass filter to EEG data.
    data shape: (n_samples, n_channels, n_times)
    """
    nyq = 0.5 * fs
    low = lowcut / nyq
    high = highcut / nyq
    b, a = butter(order, [low, high], btype='band')
    
    filtered_data = np.zeros_like(data)
    for i in range(data.shape[0]):
        for ch in range(data.shape[1]):
            filtered_data[i, ch, :] = filtfilt(b, a, data[i, ch, :])
    
    return filtered_data

eo_train_raw, eo_val_raw, eo_test_raw = split_raw_by_time(eo_data)
ec_train_raw, ec_val_raw, ec_test_raw = split_raw_by_time(ec_data)

In [4]:
def process_and_save_data(raw_train, raw_val, raw_test, prefix, window_size, stride, sfreq, output_dir):
    """Process EEG data: windowing, filtering, normalization, and save."""
    # Sliding windows
    X_train, y_train = sliding_windows(raw_train, window_size, stride)
    X_val, y_val = sliding_windows(raw_val, window_size, stride)
    X_test, y_test = sliding_windows(raw_test, window_size, stride)
    
    # Convert to numpy arrays
    X_train, X_val, X_test = np.array(X_train), np.array(X_val), np.array(X_test)
    y_train, y_val, y_test = np.array(y_train), np.array(y_val), np.array(y_test)
    
    print(f"{prefix} train/val/test: {X_train.shape}, {X_val.shape}, {X_test.shape}")
    
    # Butterworth bandpass filter (4-40 Hz, order 5)
    X_train = butter_bandpass_filter(X_train, lowcut=4, highcut=40, fs=sfreq, order=5)
    X_val = butter_bandpass_filter(X_val, lowcut=4, highcut=40, fs=sfreq, order=5)
    X_test = butter_bandpass_filter(X_test, lowcut=4, highcut=40, fs=sfreq, order=5)
    
    # Z-score normalization
    X_train = zscore(X_train, axis=2)
    X_val = zscore(X_val, axis=2)
    X_test = zscore(X_test, axis=2)
    
    # Save to disk
    suffix = f"{str(window_size).replace('.', '')}_{str(stride).replace('.', '')}"
    for name, data in [('X_train', X_train), ('X_val', X_val), ('X_test', X_test),
                       ('y_train', y_train), ('y_val', y_val), ('y_test', y_test)]:
        np.save(output_dir / f'{name.replace("_", f"_{prefix.lower()}_", 1)}_{suffix}.npy', data)
    
    return X_train, y_train

# Setup
sfreq = eo_data[0].info['sfreq']
print(f"Sampling frequency: {sfreq} Hz")

PREPROCESSED_PATH = BASE_PATH + os.getenv("PREPROCESSED_PATH")
PREPROCESSED_DIR = Path(PREPROCESSED_PATH)
PREPROCESSED_DIR.mkdir(exist_ok=True)

for window_size, stride in WINDOWS_SETUP:
    print(f"\n--- Window: {window_size}s, Stride: {stride}s ---")
    
    X_eo_train, y_eo_train = process_and_save_data(
        eo_train_raw, eo_val_raw, eo_test_raw, "EO", window_size, stride, sfreq, PREPROCESSED_DIR
    )
    X_ec_train, y_ec_train = process_and_save_data(
        ec_train_raw, ec_val_raw, ec_test_raw, "EC", window_size, stride, sfreq, PREPROCESSED_DIR
    )

Sampling frequency: 160.0 Hz

--- Window: 1s, Stride: 0.5s ---
EO train/val/test: (10355, 64, 160), (1199, 64, 160), (1199, 64, 160)
EC train/val/test: (10355, 64, 160), (1199, 64, 160), (1199, 64, 160)

--- Window: 1s, Stride: 1s ---
EO train/val/test: (5232, 64, 160), (654, 64, 160), (654, 64, 160)
EC train/val/test: (5232, 64, 160), (654, 64, 160), (654, 64, 160)

--- Window: 1.5s, Stride: 0.5s ---
EO train/val/test: (10246, 64, 240), (1090, 64, 240), (1090, 64, 240)
EC train/val/test: (10246, 64, 240), (1090, 64, 240), (1090, 64, 240)

--- Window: 1.5s, Stride: 1s ---
EO train/val/test: (5123, 64, 240), (545, 64, 240), (545, 64, 240)
EC train/val/test: (5123, 64, 240), (545, 64, 240), (545, 64, 240)

--- Window: 1.5s, Stride: 1.5s ---
EO train/val/test: (3488, 64, 240), (436, 64, 240), (436, 64, 240)
EC train/val/test: (3488, 64, 240), (436, 64, 240), (436, 64, 240)

--- Window: 2s, Stride: 0.5s ---
EO train/val/test: (10137, 64, 320), (981, 64, 320), (981, 64, 320)
EC train/val/te

In [5]:
print("Proses Selesai! Data siap masuk Embedding Model.")
print("Verifikasi Mean (harus ~0):", np.mean(X_eo_train[0, 0, :]))
print("Verifikasi Std (harus 1):", np.std(X_eo_train[0, 0, :]))

Proses Selesai! Data siap masuk Embedding Model.
Verifikasi Mean (harus ~0): -3.608224830031759e-17
Verifikasi Std (harus 1): 0.9999999999999999
